# Signal Idea 2: Weather Shock Transmission to Commodity/Equity Baskets

Research notebook capturing one implementable signal concept from quant literature (non-HFT, not price-only).


In [ ]:
# Setup: imports and robust paths
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES_ROOT = [CWD, CWD.parent]

for root in CANDIDATES_ROOT:
    if (root / "config" / "settings.py").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def resolve_first_existing(paths: list[Path], fallback: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return fallback

DATA_DIR = resolve_first_existing(
    [root / "data" / "factors" for root in CANDIDATES_ROOT],
    PROJECT_ROOT / "data" / "factors",
)
DUCKDB_PATH = resolve_first_existing(
    [root / "data" / "factors" / "factors.duckdb" for root in CANDIDATES_ROOT],
    DATA_DIR / "factors.duckdb",
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"DuckDB path  : {DUCKDB_PATH}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"DuckDB exists: {DUCKDB_PATH.exists()}")
if not DUCKDB_PATH.exists():
    print("⚠️ DuckDB file missing. Use parquet files directly for prototype tests.")


In [ ]:
from IPython.display import display
# Quick local data inventory for feasibility
candidates = [
    DATA_DIR / "prices.parquet",
    DATA_DIR / "factors_price.parquet",
    DATA_DIR / "factors_all.parquet",
    DATA_DIR / "macro.parquet",
    DATA_DIR / "macro_z.parquet",
]

rows = []
for p in candidates:
    rows.append({
        "file": p.name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / (1024**2), 2) if p.exists() else None,
    })

display_df = pd.DataFrame(rows)
display_df


## What this idea is
Create a weather-anomaly factor that links temperature/precipitation shocks to sector or commodity returns (energy, utilities, agriculture, transports).

Not price-only: signal combines weather surprise + exposure mapping by sector sensitivity.

Example signal:
- `weather_z(region, week)` projected to tradeable baskets (XLE, XLU, DBA, transport names).


## Paper lineage
- Cao and Wei (2005), *Stock Market Returns: A Note on Temperature Anomaly*.
- Various climate-finance papers on weather shocks and commodity demand/supply channels.

Why relevant: medium-horizon, macro-physical driver with intuitive economic mechanism.


## Feasibility in this project
**Feasibility: Medium-High**

- Core backtest infra already exists; only exogenous weather features are new.
- Can start with weekly rebalancing and broad ETFs before stock-level mapping.
- Low model complexity at first: rolling z-score and linear rank signal.


## Data requirements and availability
- **External required**: historical weather by region.

Easy/free sources:
- NOAA Climate Data Online (US stations/aggregates).
- Meteostat API (global daily weather, easy Python access).

Join strategy:
1. Build national/region weather anomalies.
2. Map region to sectors/commodities with fixed weights.
3. Merge with local prices parquet for testing.


## Minimal prototype notebook cell
```python
# pseudo-outline
# 1) Pull weekly temp anomaly series (NOAA/Meteostat)
# 2) Build weather shock z-score
# 3) Create long/short sector basket signal
# 4) Backtest weekly with transaction costs
```


## Recommended next experiment in this repo
1. Build a minimal feature table using currently available parquet files.
2. Run a simple walk-forward backtest using existing portfolio metrics.
3. Add one external dataset only if it materially improves explanatory power.
